# Passwords and password policies  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 4.2.a: Passwords and password policies

**Skill:** Mitigate Risk · **Maps to:** 4.2.A · **Language:** Python 3 (pytest)

## Problem
A system administrator has defined a password policy (minimum length, complexity requirements,
a banned/breached-password list, and a history of previously used passwords). Build the checker
that evaluates a candidate password against that policy and reports every rule it violates, so
the login form can show the user exactly what to fix.

## Function contract
Implement `validate_password(password: str, policy: dict) -> list[str]` in the code editor
(see the Reference section below).

## Input
- `password`, a string, the candidate password.
- `policy`, a dict. Example:
  ```python
  policy = {
      "min_length": 12,
      "require_upper": True,
      "require_lower": True,
      "require_digit": True,
      "require_symbol": True,
      "banned_list": ["password123", "letmein", "qwerty123456"],
      "history": ["OldPass2023!", "OldPass2024!"],
  }
  ```
  Any key may be omitted; a missing key means that rule is not enforced (see the Reference section below for
  exact defaults).

## Output
A `list[str]` of violated rule names, in the fixed order defined in the Reference section below:
`too_short`, `no_upper`, `no_lower`, `no_digit`, `no_symbol`, `in_banned_list`,
`reused_password`. An empty list `[]` means the password is fully compliant.

## Examples
```
validate_password("abc", {"min_length": 12, "require_upper": True, "require_digit": True})
    -> ["too_short", "no_upper", "no_digit"]

validate_password("Password123", {"banned_list": ["password123"]})
    -> ["in_banned_list"]

validate_password(
    "Tr0ub4dor&3xtra!",
    {"min_length": 12, "require_upper": True, "require_lower": True,
     "require_digit": True, "require_symbol": True,
     "banned_list": ["password123"], "history": ["OldPass2023!"]},
) -> []
```

## Constraints
- Deterministic, offline, standard library only.
- Missing policy keys are treated as "not required" (see the Reference section below for exact defaults).
- Violated rule names must be returned in the fixed order given above, even when several rules
  are violated at once.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def validate_password(password: str, policy: dict) -> list[str]
```

### Input (`password`, `policy`)
- `password`, a string, the candidate password. May be empty.
- `policy`, a dict describing the policy. All keys are optional; a missing key is treated as
  "not required" / empty. Recognized keys and defaults:
  - `min_length` (int, default `0`), minimum required length.
  - `require_upper` (bool, default `False`), require at least one uppercase letter (`str.isupper()`).
  - `require_lower` (bool, default `False`), require at least one lowercase letter (`str.islower()`).
  - `require_digit` (bool, default `False`), require at least one digit (`str.isdigit()`).
  - `require_symbol` (bool, default `False`), require at least one character that is not
    alphanumeric (`not c.isalnum()`, e.g. `! @ # $ % - _ space`).
  - `banned_list` (list[str], default `[]`), forbidden/breached passwords. Matched
    case-insensitively against `password` (e.g. `"Password1"` matches a banned entry `"password1"`).
  - `history` (list[str], default `[]`), previously used passwords for this account. Matched
    case-sensitively, exact string equality, against `password`.

### Output (`result`)
A `list[str]` of violated rule names. Only rules that are actually violated appear in the list.
The list MUST be built by checking rules in this exact fixed order, and any violated rule name
is appended in that order (this order is part of the contract, tests assert exact list equality):

1. `"too_short"`, `len(password) < policy["min_length"]`
2. `"no_upper"`, `require_upper` is true and no character in `password` is uppercase
3. `"no_lower"`, `require_lower` is true and no character in `password` is lowercase
4. `"no_digit"`, `require_digit` is true and no character in `password` is a digit
5. `"no_symbol"`, `require_symbol` is true and no character in `password` is a non-alphanumeric symbol
6. `"in_banned_list"`, `password.lower()` equals (case-insensitively) any entry in `banned_list`
7. `"reused_password"`, `password` exactly matches (case-sensitive) any entry in `history`

If none of the above are violated, return `[]` (fully compliant password).

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- `policy` may omit any subset of keys; missing keys use the defaults above.
- An empty `password` (`""`) is valid input and is evaluated normally against the rules
  (it will violate `too_short` whenever `min_length > 0`, and any `require_*` rule that is on,
  since an empty string contains no uppercase/lowercase/digit/symbol characters).
- Rule checks are independent, a password can violate multiple rules simultaneously, and all
  violated rules are reported (not just the first).
- Returns a `list[str]`, never `None`, never a set or tuple.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 4.2.a. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""


def validate_password(password: str, policy: dict) -> list:
    """Validate a candidate password against a policy dict.

    Args:
        password: The candidate password string.
        policy: A dict describing the policy. Recognized keys (all optional):
            min_length (int, default 0)          — minimum required length.
            require_upper (bool, default False)  — require an uppercase letter.
            require_lower (bool, default False)  — require a lowercase letter.
            require_digit (bool, default False)  — require a digit.
            require_symbol (bool, default False) — require a non-alphanumeric symbol.
            banned_list (list[str], default [])  — forbidden/breached passwords,
                matched case-insensitively.
            history (list[str], default [])      — previously used passwords, matched
                case-sensitively (exact match) to prevent reuse.

    Returns:
        A list of violated rule-name strings, in this fixed order:
        ["too_short", "no_upper", "no_lower", "no_digit", "no_symbol",
         "in_banned_list", "reused_password"]. Only the names of violated rules
        are included. Returns [] if the password is fully compliant.
    """
    min_length = policy.get("min_length", 0)
    require_upper = policy.get("require_upper", False)
    require_lower = policy.get("require_lower", False)
    require_digit = policy.get("require_digit", False)
    require_symbol = policy.get("require_symbol", False)
    banned_list = policy.get("banned_list", [])
    history = policy.get("history", [])

    violations = []

    if len(password) < min_length:
        violations.append("too_short")

    if require_upper and not any(c.isupper() for c in password):
        violations.append("no_upper")

    if require_lower and not any(c.islower() for c in password):
        violations.append("no_lower")

    if require_digit and not any(c.isdigit() for c in password):
        violations.append("no_digit")

    if require_symbol and not any(not c.isalnum() for c in password):
        violations.append("no_symbol")

    if any(password.lower() == banned.lower() for banned in banned_list):
        violations.append("in_banned_list")

    if any(password == prior for prior in history):
        violations.append("reused_password")

    return violations


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases (from prompt.md) --------------------------------------
def test_example_1_multiple_violations():
    result = impl.validate_password(
        "abc", {"min_length": 12, "require_upper": True, "require_digit": True}
    )
    assert result == ["too_short", "no_upper", "no_digit"]


def test_example_2_banned_list():
    result = impl.validate_password("Password123", {"banned_list": ["password123"]})
    assert result == ["in_banned_list"]


def test_example_3_fully_compliant():
    policy = {
        "min_length": 12,
        "require_upper": True,
        "require_lower": True,
        "require_digit": True,
        "require_symbol": True,
        "banned_list": ["password123"],
        "history": ["OldPass2023!"],
    }
    result = impl.validate_password("Tr0ub4dor&3xtra!", policy)
    assert result == []


# ---- hidden / edge cases ---------------------------------------------------
def test_edge_empty_password_with_requirements():
    policy = {
        "min_length": 8,
        "require_upper": True,
        "require_lower": True,
        "require_digit": True,
        "require_symbol": True,
    }
    result = impl.validate_password("", policy)
    assert result == ["too_short", "no_upper", "no_lower", "no_digit", "no_symbol"]


def test_edge_empty_policy_dict():
    # No keys at all -> everything defaults to "not required" -> compliant.
    assert impl.validate_password("anything", {}) == []
    assert impl.validate_password("", {}) == []


def test_reused_password_case_sensitive():
    policy = {"history": ["OldPass2023!", "OldPass2024!"]}
    assert impl.validate_password("OldPass2023!", policy) == ["reused_password"]
    # Different case does not count as reused (history is case-sensitive exact match).
    assert impl.validate_password("oldpass2023!", policy) == []


def test_banned_list_case_insensitive():
    policy = {"banned_list": ["LetMeIn"]}
    assert impl.validate_password("letmein", policy) == ["in_banned_list"]
    assert impl.validate_password("LETMEIN", policy) == ["in_banned_list"]


def test_require_symbol_definition():
    policy = {"require_symbol": True}
    assert impl.validate_password("abcXYZ123", policy) == ["no_symbol"]
    assert impl.validate_password("abcXYZ123!", policy) == []
    assert impl.validate_password("abc XYZ123", policy) == []  # space counts as a symbol


def test_exact_ordering_multiple_violations():
    # Violates every rule at once; asserts EXACT order per interface.md.
    policy = {
        "min_length": 20,
        "require_upper": True,
        "require_lower": True,
        "require_digit": True,
        "require_symbol": True,
        "banned_list": ["short"],
        "history": ["short"],
    }
    result = impl.validate_password("short", policy)
    assert result == [
        "too_short",
        "no_upper",
        "no_digit",
        "no_symbol",
        "in_banned_list",
        "reused_password",
    ]


def test_only_min_length_enforced():
    policy = {"min_length": 6}
    assert impl.validate_password("abc", policy) == ["too_short"]
    assert impl.validate_password("abcdef", policy) == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
